## 1단계

In [1]:
expenses=[
    {"date": "2026-09-01", "category": "식비", "amount": 8000},
    {"date": "2026-09-01", "category": "교통", "amount": 1550},
    {"date": "2026-09-02", "category": "식비", "amount": 12000},
]

# 1. 전체 순회하며 출력
for e in expenses:
    print(f"{e['date']} | {e['category']} | {e['amount']}원" )

# 2. 전체 합계
total = sum(e['amount'] for e in expenses)
print(f"\n총 지출: {total}원")

# 3. 카테고리별 합계 (group by 아직 쓰지 않고 파이썬으로만)
category_totals = {}
for e in expenses:
    cat = e["category"]
    category_totals[cat] = category_totals.get(cat, 0) + e["amount"]

print("\n카테고리별 합계: ")
for cat, amt in category_totals.items():
    print(f"  {cat}: {amt}원")

2026-09-01 | 식비 | 8000원
2026-09-01 | 교통 | 1550원
2026-09-02 | 식비 | 12000원

총 지출: 21550원

카테고리별 합계: 
  식비: 20000원
  교통: 1550원


## 2단계 입력 받아서 지출 하나 만들기

In [ ]:
expenses=[
    {"date": "2026-09-01", "category": "식비", "amount": 8000},
    {"date": "2026-09-01", "category": "교통", "amount": 1550},
]

def add_expense(expense):
    date = input("날짜 (예: 2026-09-03): ").strip()
    category = input("카테고리 (예: 식비, 교통): ").strip()

    while True:
        amount_input = input("금액: ").strip()
        try:
            amount = int(amount_input)
            break # 변환 성공했으면 반복 탈출
        except ValueError:
            print("숫자만 입력해주세요. 다시 시도합니다.")
    expenses.append({"date": date, "category": category, "amount " :amount})
    print(f"추가됨: {date} | {category} | {amount}원\n")

add_expense(expenses)
add_expense(expenses)

print("현재 expenses:")
for e in expenses:
    print(e)

## 3단계 목표: 함수를 역할별로 분리해보고, 왜 나누는지 체감하기.

In [ ]:
# 3단계: 함수 역할 분리

expenses = [
    {"date": "2026-09-01", "category": "식비", "amount": 8000},
    {"date": "2026-09-01", "category": "교통", "amount": 1500},
]

def add_expense(expenses):
    date = input("날짜 (예: 2026-09-03): ").strip()
    category = input("카테고리 (예: 식비, 교통): ").strip()

    while True:
        amount_input = input("금액: ").strip()
        try:
            amount = int(amount_input)
            break
        except ValueError:
            print("숫자만 입력해주세요. 다시 시도합니다.")

    expenses.append({"date": date, "category": category, "amount": amount})
    print(f"추가됨: {date} | {category} | {amount}원\n")


def list_expenses(expenses):
    if not expenses:
        print("등록된 지출이 없습니다.\n")
        return

    for e in expenses:
        print(f"{e['date']} | {e['category']} | {e['amount']}원")
    print()


def total_expense(expenses):
    total = sum(e["amount"] for e in expenses)
    print(f"총 지출: {total}원\n")
    return total

# 함수들 개별 테스트
list_expenses(expenses)
add_expense(expenses)
list_expenses(expenses)
total_expense(expenses)

## 4단계 목표: 메모리의 list[dict] <-> 디스크의 CSV 왕복을 이해하기

In [ ]:
# 4단계: CSV 저장/복원
import csv
from pathlib import Path

DATA_PATH = Path(__file__).resolve().parent.parent / "data" / "expenses.csv"
FIELDNAMES = ["date", "category", "description", "amount"]

def save_expenses(expenses, path=DATA_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)  # data/ 폴더 없으면 생성
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(expenses)
    print(f"{len(expenses)}건 저장 완료: {path}\n")

def load_expenses(path=DATA_PATH):
    try:
        with open(path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            expenses = []
            for row in reader:
                row["amount"] = int(row["amount"])  # CSV는 다 문자열로 읽힘!
                expenses.append(row)
            return expenses
    except FileNotFoundError:
        print(f"{path} 파일이 없습니다. 빈 목록으로 시작합니다.\n")
        return []

# --- 테스트 ---
expenses = load_expenses()          # 처음엔 파일 없으니 빈 리스트
expenses.append({"date": "2026-09-01", "category": "식비", "description": "점심", "amount": 8000})
expenses.append({"date": "2026-09-01", "category": "교통", "description": "버스", "amount": 1500})

save_expenses(expenses)             # data/expenses.csv 로 저장

reloaded = load_expenses()          # 다시 읽어옴
print("다시 불러온 데이터:")
for e in reloaded:
    print(e, type(e["amount"]))     # amount가 int인지 꼭 확인

## 5단계 전체 통합

In [ ]:
# expense_challenge.py (5단계: 전체 통합)
import csv
from pathlib import Path

DATA_PATH = Path(__file__).resolve().parent / "data" / "expenses.csv"
FIELDNAMES = ["date", "category", "description", "amount"]


def load_expenses(path=DATA_PATH):
    try:
        with open(path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            expenses = []
            for row in reader:
                row["amount"] = int(row["amount"])
                expenses.append(row)
            return expenses
    except FileNotFoundError:
        print(f"{path} 파일이 없습니다. 빈 목록으로 시작합니다.\n")
        return []


def save_expenses(expenses, path=DATA_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(expenses)
    print(f"{len(expenses)}건 저장 완료: {path}\n")


def add_expense(expenses):
    date = input("날짜 (예: 2026-09-03): ").strip()
    category = input("카테고리 (예: 식비, 교통): ").strip()
    description = input("내용 (예: 점심, 버스): ").strip()

    while True:
        amount_input = input("금액: ").strip()
        try:
            amount = int(amount_input)
            break
        except ValueError:
            print("숫자만 입력해주세요. 다시 시도합니다.")

    expenses.append({
        "date": date,
        "category": category,
        "description": description,
        "amount": amount,
    })
    print(f"추가됨: {date} | {category} | {description} | {amount}원\n")


def list_expenses(expenses):
    if not expenses:
        print("등록된 지출이 없습니다.\n")
        return

    for e in expenses:
        print(f"{e['date']} | {e['category']} | {e['description']} | {e['amount']}원")
    print()


def total_expense(expenses):
    total = sum(e["amount"] for e in expenses)
    print(f"총 지출: {total}원\n")
    return total


def print_menu():
    print("=== 지출 관리 ===")
    print("1. 지출 추가")
    print("2. 지출 목록 보기")
    print("3. 총 지출 보기")
    print("4. 저장하고 종료")
    print("5. 저장 안 하고 종료")


def main():
    expenses = load_expenses()

    while True:
        print_menu()
        choice = input("선택: ").strip()

        if choice == "1":
            add_expense(expenses)
        elif choice == "2":
            list_expenses(expenses)
        elif choice == "3":
            total_expense(expenses)
        elif choice == "4":
            save_expenses(expenses)
            print("저장하고 종료합니다.")
            break
        elif choice == "5":
            print("저장하지 않고 종료합니다.")
            break
        else:
            print("잘못된 선택입니다. 다시 입력해주세요.\n")
            continue


if __name__ == "__main__":
    main()

## 6단계 pandas 분석

In [ ]:
# analysis/analyze.py (6단계: pandas 분석)

from pathlib import Path
import pandas as pd

DATA_PATH = Path(__file__).resolve().parent.parent / "data" / "expenses.csv"

df = pd.read_csv(DATA_PATH)

print("=== 원본 데이터 ===")
print(df)
print()

# 1) 전체 합계
total = df["amount"].sum()
print(f"총 지출: {total}원\n")

# 2) 카테고리별 합계
category_totals = df.groupby("category")["amount"].sum()
print("=== 카테고리별 합계 ===")
print(category_totals)
print()

# 3) 날짜별 합계 (덤)
date_totals = df.groupby("date")["amount"].sum()
print("=== 날짜별 합계 ===")
print(date_totals)

3단계에서는 카테고리별 합계를 만들었는데

category_totals = {}
for e in expenses:
    cat = e["category"]
    category_totals[cat] = category_totals.get(cat, 0) + e["amount"]

이런식으로

category_totals = df.groupby("category")["amount"].sum()
pandas에서는 단 한줄이다

CLI 버전 (expense_challenge.py)	웹 버전 (app.py)

input()으로 값 받음	request.get_json()으로 JS가 보낸 데이터 받음

try/except ValueError (콘솔에 재입력 요청)	try/except → HTTP 400 에러 응답

print()로 결과 보여줌	jsonify()로 JSON 응답

while True 메뉴 루프	각 @app.route가 메뉴 항목 하나에 대응